# 07 · Reto CoCo: arregla una fuga de datos con un agente
> **Workshop Tec de Monterrey · De los datos a los agentes**: Abarrotera del Centro S.A. (ficticia).

**Pregunta de negocio:** un gerente del Bajío que, además, tiene el rol de director como *rol secundario* está viendo las ventas de **todas** las regiones. ¿Puedes encontrar y corregir la falla sin ser experto en políticas de acceso?

**El reto:**
1. Sembramos la falla: regresamos la Row Access Policy a una versión con `IS_ROLE_IN_SESSION()`.
2. Comprobamos la fuga.
3. Le pides a **CoCo** que la corrija (prompt exacto abajo).
4. La celda de validación da **PASS** o **FAIL**.

Prerrequisito: haber corrido `00` a `03`.

In [ ]:
%%sql -r ctx_base_esquema
-- Contexto de ejecución (primera celda, requerido en runtimes < 2.4)
USE ROLE TEC_ADMIN;
USE WAREHOUSE TEC_WH;
USE DATABASE TEC_DEMO;
USE SCHEMA TEC_DEMO.ANALITICA;

## 1 · Sembrar la falla
Esta es la versión ingenua: `IS_ROLE_IN_SESSION` también es verdadero para los roles **secundarios** del usuario.

In [ ]:
%%sql -r sembrar_falla
ALTER ROW ACCESS POLICY TEC_DEMO.ANALITICA.RAP_REGION SET BODY ->
  IS_ROLE_IN_SESSION('DIRECTOR_COMERCIAL') OR IS_ROLE_IN_SESSION('TEC_ADMIN')
  OR EXISTS (SELECT 1 FROM TEC_DEMO.ANALITICA.ACCESO_REGION a WHERE a.region = r AND IS_ROLE_IN_SESSION(a.rol))

## 2 · Antes: comprobar la fuga
Simulamos a GERENTE_BAJIO con DIRECTOR_COMERCIAL como rol secundario (así se ve una sesión con `DEFAULT_SECONDARY_ROLES = ALL`).

In [ ]:
%%sql -r antes
EXECUTE USING POLICY_CONTEXT(CURRENT_ROLE => 'GERENTE_BAJIO',
    SNOWFLAKE$SESSION_ACTIVATED_ROLES => ('GERENTE_BAJIO','DIRECTOR_COMERCIAL'))
  AS SELECT COUNT(DISTINCT region) AS regiones_visibles, LISTAGG(DISTINCT region, ', ') AS regiones
     FROM TEC_DEMO.ANALITICA.VENTAS_MENSUALES

**Lectura:** `regiones_visibles = 5`. El gerente del Bajío ve todo el país. Eso es la fuga.

## 3 · Pídele a CoCo que la corrija
Abre el panel de **CoCo** (en Snowsight o en Cortex Code) y pega exactamente:

```
En TEC_DEMO.ANALITICA la policy RAP_REGION deja ver otras regiones a GERENTE_BAJIO cuando tiene roles secundarios. Corrígela para que use CURRENT_ROLE() y la tabla ACCESO_REGION, sin desasignarla de las tablas, y ejecuta la celda de validación de este notebook.
```

CoCo debe proponer un `ALTER ROW ACCESS POLICY ... SET BODY` que use `CURRENT_ROLE()`. Revísalo, apruébalo y deja que ejecute la celda de validación.

## 4 · Validación PASS / FAIL
Revisa tres personas (todas con DIRECTOR_COMERCIAL como rol secundario) y que la política siga adjunta a las 8 tablas (6 agregados + VOZ_CLIENTE + TIENDAS_GEO).

In [ ]:
from snowflake.snowpark.context import get_active_session
s = get_active_session()
TODAS = ["Bajío", "Centro", "Norte", "Sur", "Sureste"]
casos = [("GERENTE_BAJIO", ["Bajío"]), ("GERENTE_NORTE", ["Norte"]), ("DIRECTOR_COMERCIAL", TODAS)]
ok = True
for rol, esperado in casos:
    q = (f"EXECUTE USING POLICY_CONTEXT(CURRENT_ROLE => '{rol}', "
         f"SNOWFLAKE$SESSION_ACTIVATED_ROLES => ('{rol}','DIRECTOR_COMERCIAL')) "
         "AS SELECT DISTINCT region FROM TEC_DEMO.ANALITICA.VENTAS_MENSUALES ORDER BY 1")
    visto = [r[0] for r in s.sql(q).collect()]
    paso = visto == esperado
    ok &= paso
    print(f"{'PASS' if paso else 'FAIL'} · {rol:<20} ve {visto}  (esperado {esperado})")
# 8 tablas dinámicas (03) + PRONOSTICO_MENSUAL (03b)
n = s.sql("SELECT COUNT(*) FROM TABLE(TEC_DEMO.INFORMATION_SCHEMA.POLICY_REFERENCES("
          "POLICY_NAME => 'TEC_DEMO.ANALITICA.RAP_REGION'))").collect()[0][0]
paso = n == 9
ok &= paso
print(f"{'PASS' if paso else 'FAIL'} · RAP_REGION adjunta a {n} de 9 tablas")
print("\n" + ("RETO SUPERADO" if ok else "AÚN NO: revisa la política y vuelve a correr esta celda"))

## Respaldo del instructor
Si CoCo no está disponible (red, cuota), descomenta y ejecuta la celda siguiente; después vuelve a correr la validación.

In [ ]:
%%sql -r solucion_respaldo
-- ALTER ROW ACCESS POLICY TEC_DEMO.ANALITICA.RAP_REGION SET BODY ->
--   EXISTS (SELECT 1 FROM TEC_DEMO.ANALITICA.ACCESO_REGION a
--           WHERE a.rol = CURRENT_ROLE() AND (a.region = '*' OR a.region = r))

## Lectura
- `CURRENT_ROLE()` evalúa **solo el rol activo**; `IS_ROLE_IN_SESSION()` incluye los secundarios.
- La tabla `ACCESO_REGION` es la única fuente de verdad, y la política es *fail-closed*.
- El agente de CoWork hereda esta corrección automáticamente: no hay que tocar el agente.